# Module 4: Evaluation & Capstone App

## Learning Objectives

By the end of this module, you will:
- Evaluate RAG system quality using multiple metrics
- Build a deployable Gradio web application
- Understand production considerations for RAG systems

**Prerequisites:** Complete Modules 1-3

## Setup

In [1]:
import os
import sys
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"
EVAL_PATH = PROJECT_ROOT / "evaluation" / "test_questions.json"

sys.path.insert(0, str(SRC_PATH))

from vectorstore import VectorStore
from retriever import Retriever
from rag import SciPyRAG, create_rag_system

In [2]:
# Load the RAG system
rag = create_rag_system(chroma_path=str(CHROMA_PATH)) # uses default embeddings setting (OpenAIEmbeddings)
print(f"RAG system loaded with {rag.vector_store.count()} documents")

RAG system loaded with 7698 documents


---

## 1. RAG Evaluation

Evaluating RAG systems requires measuring both **retrieval** and **generation** quality.

### Evaluation Metrics

| Category | Metric | Description |
|----------|--------|-------------|
| **Retrieval** | Precision@K | % of retrieved docs that are relevant |
| **Retrieval** | Recall@K | % of relevant docs that are retrieved |
| **Retrieval** | MRR | Mean Reciprocal Rank - how high is the first relevant result? |
| **Generation** | Faithfulness | Is the answer supported by the context? |
| **Generation** | Relevance | Does the answer address the question? |
| **Generation** | Correctness | Is the answer factually correct? |

### 1.1 Loading the Evaluation Dataset

In [3]:
if not EVAL_PATH.exists():
    raise FileNotFoundError(
        f"Evaluation file not found: {EVAL_PATH}. "
        "Make sure you're running this notebook from inside the workshop repo."
    )

with open(EVAL_PATH, "r") as f:
    eval_data = json.load(f)

questions = eval_data["questions"]
reference_answers = eval_data["reference_answers"]

print(f"Loaded {len(questions)} evaluation questions")
print(f"\nSample question:")
print(f"  Q: {questions[0]['question']}")
print(f"  Expected: {questions[0]['expected_functions']}")
print(f"  Difficulty: {questions[0]['difficulty']}")


Loaded 12 evaluation questions

Sample question:
  Q: How do I minimize a function in SciPy?
  Expected: ['minimize']
  Difficulty: easy


### 1.2 Retrieval Evaluation

In [4]:
def evaluate_retrieval(question: dict, retriever: Retriever, top_k: int = 5) -> dict:
    """
    Evaluate retrieval for a single question.

    Returns:
        Dict with precision, recall, MRR, and hit@k
    """
    # Retrieve documents
    result = retriever.retrieve(question["question"], top_k=top_k)

    # Get retrieved function names (empty string if missing)
    retrieved_functions = [r.metadata.get("function_name", "") for r in result.results]
    expected_functions = set(question["expected_functions"])

    # Basic metadata health check
    missing_fn = sum(1 for f in retrieved_functions if not f)
    if len(retrieved_functions) > 0 and missing_fn / len(retrieved_functions) >= 0.5:
        print(
            f"Warning: {missing_fn}/{len(retrieved_functions)} retrieved docs are missing "
            f"'function_name' metadata for question: {question['question'][:60]}..."
        )

    # Calculate metrics
    relevant_retrieved = sum(1 for f in retrieved_functions if f in expected_functions)
    precision = relevant_retrieved / len(retrieved_functions) if retrieved_functions else 0.0
    recall = relevant_retrieved / len(expected_functions) if expected_functions else 0.0

    # MRR (first relevant)
    mrr = 0.0
    for i, f in enumerate(retrieved_functions):
        if f in expected_functions:
            mrr = 1.0 / (i + 1)
            break

    hit_at_k = 1 if relevant_retrieved > 0 else 0

    return {
        "question": question["question"],
        "expected": list(expected_functions),
        "retrieved": retrieved_functions,
        "precision": precision,
        "recall": recall,
        "mrr": mrr,
        "hit_at_k": hit_at_k,
    }

In [5]:
# Evaluate retrieval for all questions
retriever = Retriever(rag.vector_store)

retrieval_results = []
for q in questions:
    result = evaluate_retrieval(q, retriever)
    retrieval_results.append(result)

# Calculate aggregate metrics
avg_precision = sum(r["precision"] for r in retrieval_results) / len(retrieval_results)
avg_recall = sum(r["recall"] for r in retrieval_results) / len(retrieval_results)
avg_mrr = sum(r["mrr"] for r in retrieval_results) / len(retrieval_results)
avg_hit_at_5 = sum(r["hit_at_k"] for r in retrieval_results) / len(retrieval_results)

print("Retrieval Evaluation Results")
print("=" * 50)
print(f"Average Precision@5: {avg_precision:.3f}")
print(f"Average Recall@5:    {avg_recall:.3f}")
print(f"Average MRR:         {avg_mrr:.3f}")
print(f"Hit@5:               {avg_hit_at_5:.3f}")

Retrieval Evaluation Results
Average Precision@5: 0.183
Average Recall@5:    0.917
Average MRR:         0.417
Hit@5:               0.500


In [6]:
print("\nDetailed Results:")
print("-" * 70)

for i, (q, r) in enumerate(zip(questions[:5], retrieval_results[:5])):
    status = "" if r["recall"] > 0 else ""
    print(f"\n{status} Q{i+1}: {q['question'][:50]}...")
    print(f"   Expected: {r['expected']}")
    print(f"   Retrieved: {r['retrieved'][:3]}")
    print(
        f"   Precision: {r['precision']:.2f}, "
        f"Recall: {r['recall']:.2f}, "
        f"MRR: {r['mrr']:.2f}, "
        f"Hit@5: {r['hit_at_k']}"
    )


Detailed Results:
----------------------------------------------------------------------

 Q1: How do I minimize a function in SciPy?...
   Expected: ['minimize']
   Retrieved: ['leastsq', 'minimize', 'fmin_slsqp']
   Precision: 0.40, Recall: 2.00, MRR: 0.50, Hit@5: 1

 Q2: I need to fit an exponential curve to my experimen...
   Expected: ['curve_fit']
   Retrieved: ['anderson', 'curve_fit', 'curve_fit']
   Precision: 0.60, Recall: 3.00, MRR: 0.50, Hit@5: 1

 Q3: How can I calculate the definite integral of a fun...
   Expected: ['quad']
   Retrieved: ['quad', 'nsum', 'nquad']
   Precision: 0.20, Recall: 1.00, MRR: 1.00, Hit@5: 1

 Q4: I have data points and want to interpolate values ...
   Expected: ['interp1d']
   Retrieved: ['geometric_slerp', 'cumulative_simpson', 'sampling.NumericalInverseHermite']
   Precision: 0.00, Recall: 0.00, MRR: 0.00, Hit@5: 0

 Q5: How do I solve a system of linear equations Ax = b...
   Expected: ['solve']
   Retrieved: ['solve', 'lstsq', 'cho_solve_b

### 1.3 Generation Evaluation

For generation evaluation, we can use LLM-as-judge or manual review.

In [7]:
RUN_LLM_EVAL = False  # Set True to run LLM-as-judge scoring
LLM_EVAL_MODEL = "gpt-4o-mini"

In [8]:
from generator import OpenAIGenerator
import re

def _best_effort_extract_json(text: str) -> str:
    """Extract the first JSON object found in text."""
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return m.group(0) if m else ""

def evaluate_generation_llm(
    question: str,
    generated_answer: str,
    context: str,
    reference_answer: str = None
) -> dict:
    """
    Use an LLM to evaluate generation quality.

    Returns scores for faithfulness, relevance, completeness, and code quality.
    """
    if not RUN_LLM_EVAL:
        return {
            "faithfulness": None,
            "relevance": None,
            "completeness": None,
            "code_quality": None,
            "explanation": "LLM evaluation disabled (set RUN_LLM_EVAL=True to enable)."
        }

    try:
        evaluator = OpenAIGenerator(model=LLM_EVAL_MODEL, temperature=0)
    except TypeError:
        evaluator = OpenAIGenerator(model=LLM_EVAL_MODEL)

    eval_prompt = f"""Evaluate this RAG system response.

Question: {question}

Retrieved Context (may be incomplete):
{context[:1500]}

Generated Answer:
{generated_answer}

Score the answer on these criteria (1-5 scale):
1. Faithfulness: Is the answer supported by the context? (1=contradicts context, 5=fully supported)
2. Relevance: Does the answer address the question? (1=off-topic, 5=directly answers)
3. Completeness: Is the answer thorough? (1=missing key info, 5=comprehensive)
4. Code Quality: Is any code correct and runnable? (1=broken code, 5=excellent code, null if no code)

Return ONLY valid JSON (no markdown, no extra text):
{{
  "faithfulness": <score>,
  "relevance": <score>,
  "completeness": <score>,
  "code_quality": <score or null>,
  "explanation": "<brief explanation>"
}}
"""

    result = evaluator.generate(eval_prompt, max_tokens=300)

    raw = getattr(result, "text", str(result))
    json_text = raw.strip()
    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        json_text = _best_effort_extract_json(raw)
        try:
            return json.loads(json_text)
        except Exception:
            return {
                "faithfulness": 3,
                "relevance": 3,
                "completeness": 3,
                "code_quality": None,
                "explanation": "Could not parse evaluation JSON"
            }

In [9]:
# Evaluate a sample of questions
sample_questions = questions[:3]

generation_results = []

print("Evaluating generation quality...")
print("=" * 50)

for q in sample_questions:
    # Generate answer
    response = rag.query(q['question'])
    
    # Evaluate
    scores = evaluate_generation_llm(
        question=q['question'],
        generated_answer=response.answer,
        context=response.context_used,
        reference_answer=reference_answers.get(q['id'])
    )
    
    scores['question_id'] = q['id']
    generation_results.append(scores)
    
    print(f"\nQ: {q['question'][:50]}...")
    print(f"   Faithfulness: {scores['faithfulness']}/5")
    print(f"   Relevance: {scores['relevance']}/5")
    print(f"   Completeness: {scores['completeness']}/5")
    if scores.get('code_quality'):
        print(f"   Code Quality: {scores['code_quality']}/5")

Evaluating generation quality...

Q: How do I minimize a function in SciPy?...
   Faithfulness: None/5
   Relevance: None/5
   Completeness: None/5

Q: I need to fit an exponential curve to my experimen...
   Faithfulness: None/5
   Relevance: None/5
   Completeness: None/5

Q: How can I calculate the definite integral of a fun...
   Faithfulness: None/5
   Relevance: None/5
   Completeness: None/5


In [10]:
# Aggregate generation scores
if generation_results and generation_results[0]['faithfulness'] is not None: 
    avg_faithfulness = sum(r['faithfulness'] for r in generation_results) / len(generation_results)
    avg_relevance = sum(r['relevance'] for r in generation_results) / len(generation_results)
    avg_completeness = sum(r['completeness'] for r in generation_results) / len(generation_results)
    
    print("\nGeneration Evaluation Summary")
    print("=" * 50)
    print(f"Average Faithfulness: {avg_faithfulness:.2f}/5")
    print(f"Average Relevance: {avg_relevance:.2f}/5")
    print(f"Average Completeness: {avg_completeness:.2f}/5")
else:                                                                                                                                                                                           
    print("\nGeneration Evaluation Summary")                                                                                                                                                    
    print("=" * 50)                                                                                                                                                                             
    print("LLM-as-judge evaluation disabled. Set RUN_LLM_EVAL=True to enable.")     


Generation Evaluation Summary
LLM-as-judge evaluation disabled. Set RUN_LLM_EVAL=True to enable.


### 1.4 Simple Manual Evaluation

Sometimes manual spot-checking is the most practical approach.

In [11]:
def manual_eval_checklist(question: str, answer: str, sources: list):
    """
    Display a manual evaluation checklist.
    """
    print("=" * 60)
    print(f"Question: {question}")
    print("=" * 60)
    print(f"\nAnswer:\n{answer[:500]}...")
    print(f"\nSources: {[s.metadata.get('function_name') for s in sources[:3]]}")
    print("\n" + "-" * 60)
    print("Manual Evaluation Checklist:")
    print("[ ] Answer is factually correct")
    print("[ ] Code example is runnable")
    print("[ ] Answer addresses the specific question")
    print("[ ] Appropriate level of detail")
    print("[ ] Sources are relevant")
    print("-" * 60)

# Example
test_response = rag.query("How do I minimize a function?")
manual_eval_checklist(
    "How do I minimize a function?",
    test_response.answer,
    test_response.sources
)

Question: How do I minimize a function?

Answer:
To minimize a function in Python using SciPy, you can use the `minimize` function from the `scipy.optimize` module. Below are examples demonstrating how to minimize a simple function, the Rosenbrock function, and a custom function with constraints.

### Example 1: Minimizing the Rosenbrock Function

The Rosenbrock function is a common test problem for optimization algorithms. Here's how to minimize it using the BFGS method:

```python
import numpy as np
from scipy.optimize import minimize, rosen...

Sources: ['minimize', 'elementwise.find_minimum', 'fmin_slsqp']

------------------------------------------------------------
Manual Evaluation Checklist:
[ ] Answer is factually correct
[ ] Code example is runnable
[ ] Answer addresses the specific question
[ ] Appropriate level of detail
[ ] Sources are relevant
------------------------------------------------------------


---

## 2. Building the Gradio App

Gradio makes it easy to create web interfaces for ML applications.

In [12]:
import gradio as gr
print(f"Gradio version: {gr.__version__}")

Gradio version: 6.14.0


### 2.1 Simple Interface

In [13]:
def simple_query(question: str) -> str:
    """
    Simple query function for Gradio.
    """
    if not question.strip():
        return "Please enter a question."
    
    response = rag.query(question)
    return response.answer

# Create simple interface
simple_demo = gr.Interface(
    fn=simple_query,
    inputs=gr.Textbox(label="Your Question", placeholder="How do I fit a curve in SciPy?"),
    outputs=gr.Markdown(label="Answer"),
    title="SciPy RAG Assistant",
    description="Ask questions about SciPy and get accurate, up-to-date answers.",
    examples=[
        ["How do I minimize a function?"],
        ["What's the best way to integrate a function numerically?"],
        ["How can I fit an exponential curve to my data?"]
    ],
    show_progress="full",
    submit_btn=gr.Button("Ask", variant="primary"),
    flagging_mode="never"  # Disable flagging button
)

In [14]:
# Launch (set share=True for public URL)
# simple_demo.launch()

### 2.2 Advanced Interface with Blocks

In [15]:
def query_with_sources(question: str, model: str, show_context: bool):
    """
    Query with additional options.
    """
    if not question.strip():
        return "Please enter a question.", "", ""
    
    # Switch model if needed
    if model == "GPT-4o-mini":
        rag.switch_llm("openai", "gpt-4o-mini")
    elif model == "Ollama (Local)":
        try:
            rag.switch_llm("ollama", "llama3.2")
        except:
            return "Ollama not available. Please use OpenAI.", "", ""
    
    # Query
    response = rag.query(question)
    
    # Format sources
    sources_md = "### Sources Used\n\n"
    for s in response.sources[:5]:
        func = s.metadata.get('function_name', 'Unknown')
        module = s.metadata.get('module', '')
        chunk_type = s.metadata.get('chunk_type', '')
        sources_md += f"- **{module}.{func}** ({chunk_type}) - Score: {s.score:.3f}\n"
    
    # Context (if requested)
    context_md = ""
    if show_context:
        context_md = f"### Retrieved Context\n\n```\n{response.context_used[:2000]}\n```"
    
    return response.answer, sources_md, context_md

In [16]:
# Create advanced interface with Blocks
with gr.Blocks(title="SciPy RAG Assistant") as advanced_demo:
    gr.Markdown(
        """
        # SciPy RAG Assistant
        
        Ask questions about SciPy and get accurate answers based on official documentation.
        """
    )
    
    with gr.Row():
        with gr.Column(scale=3):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="How do I fit a curve in SciPy?",
                lines=2
            )
        with gr.Column(scale=1):
            model_select = gr.Dropdown(
                choices=["GPT-4o-mini", "Ollama (Local)"],
                value="GPT-4o-mini",
                label="Model"
            )
            show_context = gr.Checkbox(label="Show Retrieved Context", value=False)
    
    submit_btn = gr.Button("Ask", variant="primary")
    
    with gr.Row():
        with gr.Column(scale=2):
            answer_output = gr.Markdown(label="Answer")
        with gr.Column(scale=1):
            sources_output = gr.Markdown(label="Sources")
    
    context_output = gr.Markdown(label="Context", visible=True)
    
    # Example questions
    gr.Markdown("### Example Questions")
    examples = gr.Examples(
        examples=[
            ["How do I minimize a function in SciPy?"],
            ["What's the best way to fit an exponential curve to data?"],
            ["How can I calculate a definite integral?"],
            ["I need to solve a system of linear equations Ax = b"],
            ["How do I design a Butterworth filter?"]
        ],
        inputs=question_input
    )
    
    # Connect components
    submit_btn.click(
        fn=query_with_sources,
        inputs=[question_input, model_select, show_context],
        outputs=[answer_output, sources_output, context_output],
        show_progress="full"
    )
    
    question_input.submit(
        fn=query_with_sources,
        inputs=[question_input, model_select, show_context],
        outputs=[answer_output, sources_output, context_output]
    )

In [17]:
# Test the interface function
answer, sources, context = query_with_sources(
    "How do I fit a curve?",
    "GPT-4o-mini",
    True
)

print("Answer:")
print(answer[:500])
print("\nSources:")
print(sources)

Answer:
To fit a curve using SciPy's `curve_fit` function, you need to define a model function that describes the relationship between your independent variable (`xdata`) and dependent variable (`ydata`). Here's a step-by-step guide along with a working code example.

### Step-by-Step Guide

1. **Import Necessary Libraries**: You'll need `numpy` for numerical operations and `scipy.optimize` for the `curve_fit` function. Optionally, you can use `matplotlib` for plotting.

2. **Define Your Model Function*

Sources:
### Sources Used

- **scipy.optimize.curve_fit** (summary) - Score: 0.782
- **scipy.optimize.curve_fit** (full_text) - Score: 0.779
- **scipy.optimize.curve_fit** (full_text) - Score: 0.771



### 2.3 Launch the App

Uncomment to launch the Gradio interface:

In [18]:
# Uncomment to launch the app:
# advanced_demo.launch(share=False)  # Set share=True for public URL

---

## 3. Production Considerations

### 3.1 Caching

In [19]:
from functools import lru_cache
import hashlib

# Simple in-memory cache for embeddings
class EmbeddingCache:
    def __init__(self, maxsize: int = 1000):
        self.cache = {}
        self.maxsize = maxsize
    
    def _hash(self, text: str) -> str:
        return hashlib.md5(text.encode()).hexdigest()
    
    def get(self, text: str):
        return self.cache.get(self._hash(text))
    
    def set(self, text: str, embedding: list):
        if len(self.cache) >= self.maxsize:
            # Remove oldest entry (simple FIFO)
            oldest = next(iter(self.cache))
            del self.cache[oldest]
        self.cache[self._hash(text)] = embedding

print("Caching strategy: Use LRU cache for embeddings to reduce API calls")

Caching strategy: Use LRU cache for embeddings to reduce API calls


### 3.2 Error Handling

In [20]:
import time

def robust_query(question: str, max_retries: int = 3) -> str:
    """
    Query with retry logic and error handling.
    """
    for attempt in range(max_retries):
        try:
            response = rag.query(question)
            return response.answer
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Error: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                return f"Sorry, I encountered an error: {str(e)}. Please try again."

print("Error handling: Exponential backoff with max 3 retries")

Error handling: Exponential backoff with max 3 retries


### 3.3 Keeping Documentation Updated

Documentation Update Strategies:

1. **Scheduled Re-scraping**
   - Run scraper weekly/monthly via cron job
   - Compare checksums to detect changes
   - Only re-embed changed documents

2. **Version Tracking**
   - Store SciPy version with each document
   - Filter by version in retrieval
   - Support multiple versions simultaneously

3. **Incremental Updates**
   - Monitor SciPy release notes
   - Only update changed functions
   - Keep embedding costs low

4. **Hybrid Approach**
   - Static base knowledge
   - Real-time web search for latest info
   - Merge results intelligently

## Module 4 Summary

### What We Built

1. **Evaluation Framework**: Retrieval and generation metrics
2. **Gradio App**: Interactive web interface for the RAG system
3. **Production Patterns**: Caching, error handling, updates

### Key Takeaways

- **Evaluation is essential**: Measure before optimizing
- **Multiple metrics**: Both retrieval and generation matter
- **User experience**: Gradio makes demos easy
- **Production readiness**: Consider caching, errors, updates

### Next Steps

1. Deploy your app to HuggingFace Spaces
2. Add more SciPy modules to the knowledge base
3. Explore the bonus Advanced RAG module

---

## Exercises

1. **Expand evaluation**: Add more test questions and run full evaluation
2. **Custom UI**: Add dark mode or export features
3. **Feedback System**: Add 👍/👎 buttons that log feedback to a JSON file for later analysis
4. **Deploy**: Push your app to HuggingFace Spaces or your own server
5. **Conversation History**: Maintain context from previous questions; store the last 3 Q&A pairs and include them in the prompt